# English–German handwriting-aware bigrams: meeting walkthrough

This notebook follows the proof of concept from IAM and READ handwriting to a provisional combined bigram tokenizer. It uses the real DTLR and POC files, but explains each stage in everyday language. Nothing here changes DTLR, TVA, or the frozen connectivity method.

**Pipeline:** IAM/READ image + transcription → dataset-fine-tuned DTLR character boxes → monotonic alignment → 8-connected ink components → dominant-core-v3 pair evidence → split-safe train scores → combined text tokenizer.

Before running it, set the paths in the next cell. The notebook expects the completed frozen 32-line IAM and READ validation exports, both full training-score files, and the combined model. DTLR inference itself must run in the Conda/CUDA environment on Linux/WSL with the RTX 4060.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Change these paths for the demonstration machine.
# POC_ROOT is the checkout containing poc/.
POC_ROOT = Path(os.environ.get('DTLR_POC_ROOT', Path.cwd())).resolve()
DATA_ROOT = Path(os.environ.get('DTLR_DATA_ROOT', '/absolute/path/to/dtlr-data')).expanduser()
OUTPUT_ROOT = Path(os.environ.get('DTLR_OUTPUT_ROOT', '/absolute/path/to/dtlr-output')).expanduser()
DEMO_DATASET = os.environ.get('DTLR_DEMO_DATASET', 'READ').upper()  # IAM or READ
VALIDATION_RUNS = {'IAM': 'iam-valid-32', 'READ': 'read-valid-32'}
assert DEMO_DATASET in VALIDATION_RUNS, 'DTLR_DEMO_DATASET must be IAM or READ.'
RUN_NAME = VALIDATION_RUNS[DEMO_DATASET]
DETECTIONS = OUTPUT_ROOT / RUN_NAME / 'detections.jsonl'

assert (POC_ROOT / 'poc' / 'dtlr_poc').is_dir(), 'Set DTLR_POC_ROOT to the POC checkout.'
sys.path.insert(0, str(POC_ROOT / 'poc'))
print('POC checkout:', POC_ROOT)
print('Data root:', DATA_ROOT)
print('Visual walkthrough dataset:', DEMO_DATASET)
print('Detection file:', DETECTIONS)

## 1. Start with a small, fixed set of lines

We use a small, frozen validation set so results can be inspected without changing the sample after seeing outcomes. IAM or READ supplies the correct text; DTLR only estimates where each character is on the image. The commands below show how the two selections were frozen.

In [ ]:
# These commands are shown rather than run automatically.
print(f'''cd {POC_ROOT}
python poc/scripts/freeze_iam_selection.py \
  --data-root {DATA_ROOT} --split valid --count 32 \
  --seed iam-dominant-core-v3-20260821 \
  --output {OUTPUT_ROOT}/iam-valid-32/selection.json

python poc/scripts/freeze_read_selection.py \
  --data-root {DATA_ROOT} --split valid --count 32 \
  --seed read-dominant-core-v3-transfer-20260830 \
  --output {OUTPUT_ROOT}/read-valid-32/selection.json''')

# For this live walkthrough we normally reuse an already completed small export.
# Its records include the line ID, dataset transcription, DTLR boxes, scores, and run provenance.

## 2. Read the DTLR results

Each result contains the line image, ground-truth transcription, estimated character boxes, and provenance. Generic English/German checkpoints are synthetic-language pretraining; they are not dataset localization evidence. IAM must use the IAM-fine-tuned checkpoint and READ must use the READ-fine-tuned checkpoint.

In [ ]:
EXPECTED_CHECKPOINT_KIND = {'IAM': 'iam-finetuned', 'READ': 'read-finetuned'}[DEMO_DATASET]
if not DETECTIONS.exists():
    raise FileNotFoundError(
        f'No {DEMO_DATASET} validation detections at {DETECTIONS}. '
        'Run the corresponding dataset-fine-tuned export command from poc/README.md first.'
    )

records = [json.loads(line) for line in DETECTIONS.read_text(encoding='utf-8').splitlines() if line]
assert len(records) == 32, f'Expected the complete frozen 32-line run, found {len(records)} records.'
assert {row['dataset'] for row in records} == {DEMO_DATASET}, 'Wrong validation dataset.'
assert {row['split'] for row in records} == {'valid'}, 'This walkthrough expects the frozen validation split.'
assert {row['checkpoint']['kind'] for row in records} == {EXPECTED_CHECKPOINT_KIND}, 'Use the dataset-fine-tuned checkpoint, not a generic pretrained checkpoint.'
assert len({row['checkpoint']['sha256'] for row in records}) == 1, 'Mixed checkpoint provenance.'
assert len({row['repo_commit'] for row in records}) == 1, 'Mixed repository provenance.'
selection_hashes = {row.get('selection_manifest', {}).get('sha256') for row in records}
assert len(selection_hashes) == 1 and None not in selection_hashes, 'Missing or mixed frozen-selection provenance.'
print(f'Loaded {len(records)} frozen {DEMO_DATASET} validation lines.')
print('Checkpoint:', EXPECTED_CHECKPOINT_KIND, records[0]['checkpoint']['sha256'][:12] + '…')
print('Inference repository commit:', records[0]['repo_commit'][:12])
print('Frozen selection SHA-256:', next(iter(selection_hashes)))
for row in records[:5]:
    print(f"{row['line_id']}: {row['transcription']}")
if len(records) > 5:
    print(f'… and {len(records) - 5} more lines')

## 3. Look at one line and DTLR's estimated letter locations

The green rectangles are DTLR's best guesses for character locations. They are useful guides, but not perfect borders around the letters. That is why the later ink check is needed.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

LINE_INDEX = 0  # Change this number to present another frozen validation line.
record = records[LINE_INDEX]
image = Image.open(DATA_ROOT / record['image_relpath']).convert('RGB')
fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(image)
for number, detection in enumerate(sorted(record['detections'], key=lambda item: item['box_xyxy'][0])):
    x0, y0, x1, y1 = detection['box_xyxy']
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor='lime', linewidth=1))
    ax.text(x0, max(0, y0-3), str(number), color='lime', fontsize=8, backgroundcolor='black')
ax.set_title(f"{DEMO_DATASET} transcription: {record['transcription']}")
ax.axis('off');
plt.show()

## 4. Match the correct letters to the estimated boxes

The correct text may not line up perfectly with DTLR's predictions. A fixed monotonic edit alignment associates transcription positions with ordered detections. Missing, extra, and substituted detections remain visible instead of being hidden. Connectivity can be measured when both positions have boxes, but tokenizer statistics use only pairs where both predicted characters exactly match the transcription.

In [ ]:
from dtlr_poc.alignment import gt_detection_map

detections = sorted(record['detections'], key=lambda item: item['box_xyxy'][0])
mapping = gt_detection_map(record['transcription'], [item['predicted_char'] for item in detections])
for index, character in enumerate(record['transcription']):
    item = mapping[index]
    box = 'no box' if item.detection_index is None else f'box {item.detection_index}'
    print(f'{index:>2}: {character!r:>4} → {box:>8} ({item.operation})')

## 5. Turn the writing into separate ink shapes

We change the gray image into a simple black-and-white ink picture. Then we colour every separate touching ink shape differently. Diagonal touching counts as touching. This gives us a direct, physical view of the handwriting.

In [ ]:
import numpy as np
from dtlr_poc.ccl import label_ink, otsu_threshold, pair_component_evidence

gray = np.asarray(image.convert('L'))
threshold = otsu_threshold(gray)
labels, component_count = label_ink(gray, threshold)
print(f'Automatic ink threshold: {threshold}; separate ink shapes: {component_count}')

# Build a dark, saturated display palette. This changes only the picture shown
# in the notebook; it does not alter the threshold or component labels.
from matplotlib.colors import hsv_to_rgb
palette = np.ones((component_count + 1, 3), dtype=float)  # label 0 stays white
for component in range(1, component_count + 1):
    hue = (component * 0.61803398875) % 1.0
    palette[component] = hsv_to_rgb((hue, 0.95, 0.62))
coloured_components = palette[labels]

# Remove unused vertical margins and give each view the full notebook width.
ink_rows = np.flatnonzero(np.any(labels > 0, axis=1))
if ink_rows.size:
    top = max(0, int(ink_rows[0]) - 5)
    bottom = min(gray.shape[0], int(ink_rows[-1]) + 6)
else:
    top, bottom = 0, gray.shape[0]

fig, axes = plt.subplots(2, 1, figsize=(18, 5), constrained_layout=True)
axes[0].imshow(gray[top:bottom], cmap='gray', interpolation='nearest')
axes[0].set_title('Original handwriting')
axes[1].imshow(coloured_components[top:bottom], interpolation='nearest')
axes[1].set_title('Connected components — each touching ink shape has one colour')
for ax in axes:
    ax.axis('off')
plt.show()

## 6. Ask one simple question about two neighbouring letters

Choose a pair below. The final method gives each letter its own non-overlapping inner area. It calls the pair joined only when the same *main* ink shape is largest in both areas. If the picture is unclear, the answer is `unknown`, not `separate`.

In [ ]:
# Set this to a transcription index to inspect a specific pair. None chooses
# the first non-space pair with exact alignment and usable v3 evidence.
LEFT_GT_INDEX = None
candidates = [LEFT_GT_INDEX] if LEFT_GT_INDEX is not None else range(len(record['transcription']) - 1)
selected = None
for candidate in candidates:
    left_candidate, right_candidate = mapping[candidate], mapping[candidate + 1]
    pair_candidate = record['transcription'][candidate:candidate + 2]
    if any(char.isspace() for char in pair_candidate):
        continue
    if left_candidate.detection_index is None or right_candidate.detection_index is None:
        continue
    candidate_result = pair_component_evidence(
        labels, detections[left_candidate.detection_index]['box_xyxy'],
        detections[right_candidate.detection_index]['box_xyxy'])
    exact = left_candidate.operation == right_candidate.operation == 'match'
    if LEFT_GT_INDEX is not None or (exact and candidate_result['dominant_core_usable']):
        selected = (candidate, left_candidate, right_candidate, candidate_result)
        break
if selected is None:
    raise ValueError('No suitable neighbouring pair was found in this line.')
left_index, left, right, result = selected
right_index = left_index + 1
pair = record['transcription'][left_index:right_index + 1]
answer = 'unknown' if not result['dominant_core_usable'] else ('joined' if result['connected_dominant_core_v3'] else 'separate')
print(f"Pair {pair!r} at GT positions {left_index}:{right_index}: {answer}")
print('Alignment:', left.operation, '/', right.operation)
print('Old full-box answer:', result['connected_box_intersection_v1'])
print('Final dominant-core-v3 answer:', result['connected_dominant_core_v3'])
print('If unknown, the mechanical reason is:', result['unusable_reason_codes'])

## 7. See why the final rule is safer

The old rule only asked whether any coloured ink shape appeared in both full boxes. That could be fooled by overlapping boxes. The final rule uses the two smaller inner areas shown below and asks whether the same colour is the biggest contributor on both sides.

In [ ]:
left_box = detections[left.detection_index]['box_xyxy']
right_box = detections[right.detection_index]['box_xyxy']
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.imshow(coloured_components, interpolation='nearest')
for box, colour, name in [(left_box, 'deepskyblue', 'left DTLR box'),
                          (right_box, 'orange', 'right DTLR box'),
                          (result['left_core_box'], 'blue', 'left exclusive core'),
                          (result['right_core_box'], 'red', 'right exclusive core')]:
    x0, y0, x1, y1 = box
    if x1 >= x0 and y1 >= y0:  # inverted cores are abstentions, not drawable rectangles
        ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor=colour, linewidth=2, label=name))
padding = 20
ax.set_xlim(max(0, min(left_box[0], right_box[0]) - padding),
            min(gray.shape[1], max(left_box[2], right_box[2]) + padding))
ax.set_ylim(min(gray.shape[0], max(left_box[3], right_box[3]) + padding),
            max(0, min(left_box[1], right_box[1]) - padding))
ax.set_title(f"Pair {pair!r}: same dominant component = {result['connected_dominant_core_v3']}")
ax.legend(loc='upper right')
ax.axis('off')
plt.show()
print('Largest shape in left core:', result['left_dominant_component'])
print('Largest shape in right core:', result['right_dominant_component'])

## 8. Repeat this for the small batch and summarize it

The pipeline repeats the same careful check for every neighbouring pair, writes one evidence row per pair, and then creates a separate score for each pair of letters. The following cell runs that evidence step if it has not already been done.

In [ ]:
EVIDENCE_DIR = OUTPUT_ROOT / RUN_NAME / 'bigrams-dominant-core-v3'
SCORES = EVIDENCE_DIR / 'bigram_scores.json'
if not SCORES.exists():
    import subprocess
    subprocess.run([sys.executable, str(POC_ROOT / 'poc/scripts/build_bigram_evidence.py'),
                    '--detections', str(DETECTIONS), '--data-root', str(DATA_ROOT),
                    '--output-dir', str(EVIDENCE_DIR)], check=True)
scores = json.loads(SCORES.read_text(encoding='utf-8'))
small_manifest = json.loads((EVIDENCE_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Small-batch evidence:', {key: small_manifest[key] for key in ('line_count', 'evidence_count', 'score_count', 'dominant_core_unusable_count')})
for row in sorted(scores, key=lambda row: (-row['n_exact_alignment'], row['pair']))[:12]:
    rate = row['exact_alignment_connected_rate']
    print(f"{row['pair']!r}: seen {row['n_exact_alignment']} exact times; joined rate = {rate}")

## 9. Show the combined English–German tokenizer

The combined model pools exact-alignment IAM-train and READ-train observations while retaining each dataset's counts and rates. It keeps letter pairs seen at least 20 times and connected at least half the time, then chooses the maximum-utility non-overlapping segmentation. It uses learned handwriting statistics; it does not inspect a new image at tokenization time.

In [ ]:
from dtlr_poc.tokenizer import HandwritingBigramTokenizer, build_combined_model
from hashlib import sha256

# Only full training scores are accepted; validation and test evidence remain separate.
SCORE_PATHS = {
    'IAM': OUTPUT_ROOT / 'iam-train-full/bigrams-dominant-core-v3/bigram_scores.json',
    'READ': OUTPUT_ROOT / 'read-train-full/bigrams-dominant-core-v3/bigram_scores.json',
}
MANIFEST_PATHS = {dataset: path.parent / 'manifest.json' for dataset, path in SCORE_PATHS.items()}
SAVED_MODEL = OUTPUT_ROOT / 'combined-iam-read-v1/model.json'
missing = [str(path) for path in [*SCORE_PATHS.values(), *MANIFEST_PATHS.values()] if not path.exists()]
if missing:
    raise FileNotFoundError('Missing completed training artifacts:\n' + '\n'.join(missing))
raw_scores = {dataset: path.read_bytes() for dataset, path in SCORE_PATHS.items()}
score_hashes = {dataset: sha256(raw).hexdigest() for dataset, raw in raw_scores.items()}
if SAVED_MODEL.exists():
    model = json.loads(SAVED_MODEL.read_text(encoding='utf-8'))
else:
    model = build_combined_model(
        {dataset: json.loads(raw) for dataset, raw in raw_scores.items()}, score_hashes,
        minimum_count=20, rate_threshold=0.5, unicode_normalization='NFC')
assert model['model_version'] == 'iam-read-combined-v1'
assert model['training_splits'] == {'IAM': 'train', 'READ': 'train'}
for dataset, digest in score_hashes.items():
    assert model['source_scores'][dataset]['sha256'] == digest, f'{dataset} score hash mismatch.'
tokenizer = HandwritingBigramTokenizer(model)
for dataset, path in MANIFEST_PATHS.items():
    manifest = json.loads(path.read_text(encoding='utf-8'))
    values = {key: manifest[key] for key in
              ('line_count', 'evidence_count', 'score_count', 'dominant_core_unusable_count')}
    print(f'{dataset} full-train scale:', values)
print('Model status:', model['status'])
print('Policy:', model['policy'])
print('Unicode normalization:', model['text_normalization'])
print('TVA-familiar interface: vocab, idx_token, size, load, encode, decode')
print('Tokenizer size:', tokenizer.size, '(CTC blank + characters + eligible bigrams)')
print('Eligible bigrams:', model['eligible_bigram_count'])
for text in ('the handwriting', 'für größere Wörter'):
    example = tokenizer.segment(text)
    token_ids = tokenizer.encode(text)
    rendered = ' | '.join('␠' if token == ' ' else token for token in example['tokens'])
    print(f'\nInput:  {text}\nTokens: {rendered}\nIDs:    {token_ids}\nDecoded: {tokenizer.decode(token_ids)}\nBigram tokens: {example["bigram_token_count"]}')
print('\nSelected cross-language and German examples:')
by_token = {item['token']: item for item in model['vocabulary']}
for token in ('he', 'er', 'ür', 'ör'):
    item = by_token[token]
    print(f"  {token!r}: pooled={item['n_exact_alignment_connected']}/{item['n_exact_alignment']} ({item['utility']:.3f}); by dataset={item['dataset_statistics']}")

## Meeting summary

### What has been demonstrated

- IAM and READ supply character identity through ground-truth transcription; their respective fine-tuned DTLR checkpoints supply approximate localization.
- Otsu binarization and 8-connected component labelling measure physical ink connectivity.
- Frozen `dominant-core-v3` calls a pair connected only when the same unique largest component dominates both raster-safe exclusive character cores.
- Missing detections, inverted or empty cores, and dominant ties are explicit abstentions—not silently converted to disconnected labels.
- Training, validation, and test statistics remain separate. The combined vocabulary uses only IAM **train** and READ **train** scores.
- The provisional tokenizer is handwriting-aware statistically: utilities come from pooled physical ink-connectivity observations while per-dataset evidence remains auditable. It is not image-adaptive when tokenizing new text.

### Verified scale and validation context

- IAM frozen validation: 32 lines and 884 pairs. Its stratified review had 146 evaluable v3 cases: 142 correct and four false-disconnected.
- READ frozen validation: 32 lines and 449 pairs. Its 160-case stratified review was complete; 119 cases were evaluable, with 109 correct, eight false-disconnected, and two false-connected. Of 35 unusable cases, 28 were confirmed appropriate abstentions and seven remained uncertain; none was judged an unnecessary abstention.
- These queues deliberately contain all difficult disagreements and abstentions plus deterministic agreement audits. **Their raw reviewed ratios are not population accuracy estimates.**
- Full IAM train export: 5,694 lines, 160,536 evidence rows, 1,259 aggregate pair scores, and 1,683 dominant-core abstentions.
- Full READ train export: 8,367 lines, 133,432 evidence rows, 1,136 aggregate pair scores, and 5,832 dominant-core abstentions.
- Combined model v1: 494 tokens = one CTC blank + 91 characters + 402 eligible handwriting bigrams; NFC normalization and both source-score hashes are recorded.
- Provisional policy: letters only, at least 20 exact-alignment observations, connected rate at least 0.5, with dynamic programming selecting maximum-total-utility non-overlapping bigrams. These thresholds are demonstration settings, not held-out optimized values.

### Safe conclusion

This is an end-to-end English–German proof of concept showing that physical handwriting connectivity can produce a train-derived bigram vocabulary and deterministic text segmentation. It complements the earlier forced-alignment proposal rather than invalidating it. The next scientific step is a controlled comparison against TVA's character and linguistic-bigram baselines on identical OnHW splits. No OnHW recognition improvement is claimed yet.